# SENTIMENT DEVELOPMENT NOTEBOOK 03 — Error analysis

Diagnostic analysis of the saved Experiment 01 validation predictions. This notebook does not train, retune, download models, alter labels, or access frozen-test text, labels, predictions, or metrics. It writes ID-level summaries only; raw conversational text is never loaded.

**Experiment 02 status:** NOT STARTED.


In [2]:
from pathlib import Path
import re
import json
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO = NOTEBOOK_DIR.parents[2]
SENTIMENT = REPO / 'ml' / 'sentiment'
OUT = SENTIMENT / 'outputs' / 'development'
ERR = OUT / 'error_analysis'
PLOTS = SENTIMENT / 'plots' / 'development' / 'error_analysis'
ERR.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(parents=True, exist_ok=True)
pred = pd.read_csv(OUT / 'predictions.csv')
membership = pd.read_csv(OUT / 'splits' / 'development_split_membership.csv')

required = {'record_id','human_mood','predicted_state','p_calm','p_neutral','p_distressed','confidence'}
assert required.issubset(pred.columns), f'Missing prediction columns: {required - set(pred.columns)}'
assert set(pred['language']) == {'SI'}, 'FAIL LOUDLY: non-Sinhala data accessed'
assert pred['record_id'].is_unique and pred['record_id'].notna().all(), 'Duplicate/missing prediction IDs'
assert not pred[['p_calm','p_neutral','p_distressed']].isna().any().any(), 'Missing probabilities'
assert set(pred['human_mood']) <= {'CALM','NEUTRAL','DISTRESSED'}, 'Unexpected target label'
assert set(pred['predicted_state']) <= {'CALM','NEUTRAL','DISTRESSED'}, 'Unexpected prediction label'
assert len(pred) == 18, 'Expected exactly the Experiment 01 validation predictions'
assert set(membership['split']) == {'train','validation'}, 'Unexpected split values'
assert set(pred['record_id']) == set(membership.loc[membership['split']=='validation','record_id']), 'Prediction IDs are not exactly the saved validation split'

# Read frozen IDs only as an exclusion guard; never load frozen text, labels, predictions, or metrics.
manifest = SENTIMENT / 'data' / 'processed' / 'FROZEN_TEST_SET_MANIFEST.md'
frozen_ids = {candidate for line in manifest.read_text(encoding='utf-8').splitlines() if (candidate := line.removeprefix('- ').strip()) and re.fullmatch(r'(?:SI|EN)-[A-Z0-9-]+', candidate)}
assert len(frozen_ids) == 120, 'Frozen manifest guard failed'
assert not set(pred['record_id']) & frozen_ids, 'FAIL LOUDLY: frozen-test ID accessed'
print('Verified: 18 saved Sinhala validation predictions; zero frozen-test overlap; no text loaded.')

Verified: 18 saved Sinhala validation predictions; zero frozen-test overlap; no text loaded.


In [3]:
labels = ['CALM','NEUTRAL','DISTRESSED']
pred['correct'] = pred['human_mood'] == pred['predicted_state']
pred['true_class_probability'] = pred.apply(lambda r: r['p_' + r['human_mood'].lower()], axis=1)
pred['predicted_class_probability'] = pred.apply(lambda r: r['p_' + r['predicted_state'].lower()], axis=1)
pred['error_type'] = pred.apply(lambda r: 'CORRECT' if r['correct'] else f"{r['human_mood']}-> {r['predicted_state']}", axis=1)
distribution = pd.DataFrame({'true_count': pred['human_mood'].value_counts().reindex(labels, fill_value=0), 'predicted_count': pred['predicted_state'].value_counts().reindex(labels, fill_value=0)})
confusion = pd.crosstab(pred['human_mood'], pred['predicted_state']).reindex(index=labels, columns=labels, fill_value=0)
per_class = []
for label in labels:
    tp = int(((pred['human_mood']==label) & (pred['predicted_state']==label)).sum())
    support = int((pred['human_mood']==label).sum())
    predicted = int((pred['predicted_state']==label).sum())
    recall = tp/support if support else 0.0
    precision = tp/predicted if predicted else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    per_class.append({'class':label,'correct':tp,'incorrect':support-tp,'support':support,'predicted_count':predicted,'precision':precision,'recall':recall,'f1':f1})

distribution.to_csv(ERR / 'class_distributions.csv')
confusion.to_csv(ERR / 'confusion_matrix.csv')
pd.DataFrame(per_class).to_csv(ERR / 'per_class_summary.csv', index=False)
pred[['record_id','human_mood','predicted_state','p_calm','p_neutral','p_distressed','confidence','true_class_probability','predicted_class_probability','correct','error_type']].to_csv(ERR / 'validation_error_summary.csv', index=False)
print(distribution)
print(confusion)

            true_count  predicted_count
CALM                 2                1
NEUTRAL              9                4
DISTRESSED           7               13
predicted_state  CALM  NEUTRAL  DISTRESSED
human_mood                                
CALM                0        1           1
NEUTRAL             1        3           5
DISTRESSED          0        0           7


In [4]:
# Probability summaries by true and predicted class; no raw text.
prob_cols = ['p_calm','p_neutral','p_distressed','confidence','true_class_probability','predicted_class_probability']
by_true = pred.groupby('human_mood')[prob_cols].agg(['count','mean','median','min','max']).reindex(labels)
by_pred = pred.groupby('predicted_state')[prob_cols].agg(['count','mean','median','min','max']).reindex(labels)
by_true.to_csv(ERR / 'probabilities_by_true_class.csv')
by_pred.to_csv(ERR / 'probabilities_by_predicted_class.csv')
pred[pred['human_mood']=='CALM'][['record_id','predicted_state','p_calm','p_neutral','p_distressed','confidence','correct']].to_csv(ERR / 'calm_validation_ids_only.csv', index=False)
pred[pred['human_mood'].isin(['NEUTRAL','DISTRESSED'])].groupby(['human_mood','predicted_state']).size().rename('count').to_csv(ERR / 'neutral_distressed_confusion.csv')
history = pd.DataFrame(json.loads((OUT / 'training_history.json').read_text(encoding='utf-8')))
history.to_csv(ERR / 'training_history.csv', index=False)
assert history['macro_f1'].idxmax() == 1, 'Saved best epoch is not epoch 2'
history.to_json(ERR / 'training_dynamics.json', orient='records', indent=2)
print('Saved probability, CALM ID-only, confusion, and training-dynamics summaries.')

Saved probability, CALM ID-only, confusion, and training-dynamics summaries.


## Interpretation boundary

The companion findings file must distinguish observed facts, plausible explanations, and unsupported speculation. In particular, CALM F1=0 is based on only two validation records; it does not prove that SinBERT cannot classify CALM. This notebook is diagnostic only. Do not start Experiment 02 from its output.